# ReviewIQ — Model Validation

Validates `LiYuan/amazon-review-sentiment-analysis` (BERT fine-tuned on Amazon reviews, 1–5 star output) on CPU before deploying to RHOAI single-model serving.

**Workbench:** Standard Data Science image, Python 3.11, no GPU.

## 1. Environment check

In [ ]:
import sys
print(sys.version)
print(sys.executable)

## 2. Install dependencies

`%pip` installs into the running kernel's environment. Restart the kernel after this cell completes (Kernel → Restart).

In [ ]:
%pip install --quiet "transformers[torch]" "numpy<2"

## 3. Load model and tokenizer

First run downloads ~500MB from HuggingFace into the workbench PVC (`~/.cache/huggingface`).

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

MODEL_ID = "LiYuan/amazon-review-sentiment-analysis"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID)
clf = pipeline("text-classification", model=model, tokenizer=tokenizer)

print(model.config.id2label)

## 4. Single-prediction sanity checks

In [ ]:
print(clf("Battery died after two weeks. Very disappointed."))
print(clf("Exceeded expectations, works perfectly and shipping was fast."))

## 5. Batch test with a realistic review spread

Watch the mid-range reviews — they should land at 2–3 stars, not just cluster at the extremes. That's the signal this checkpoint is usable for the analytics tier.

In [ ]:
reviews = [
    "Terrible quality, broke on day one",
    "It's okay, does the job but nothing special",
    "Absolutely love it, best purchase this year",
    "Decent for the price but the instructions were useless",
    "Stopped working after a month, support never responded",
]

for r, out in zip(reviews, clf(reviews)):
    print(f"{out['label']:>8}  {out['score']:.3f}  {r[:50]}")

## 6. Full class distribution

ReviewCraft analytics will want the full 1–5 star probability distribution, not just top-1.

In [ ]:
clf("Decent product but arrived damaged", top_k=None)

## 7. CPU latency baseline

Sets expectations before OVMS. On an m5a.4xlarge expect roughly 50–150 ms/review single-threaded; ONNX + OVMS will improve on this.

In [ ]:
import time

batch = reviews * 20  # 100 reviews
t = time.time()
_ = clf(batch)
elapsed = time.time() - t
print(f"{elapsed:.1f}s total — {elapsed/len(batch)*1000:.0f} ms/review")

## 8. Export to ONNX (next step: OVMS serving)

Produces `reviewiq-sentiment/1/model.onnx` — the `/1/` directory is the model version, the layout OVMS expects. Upload the folder to your S3/MinIO data connection, then deploy via Models → Deploy model → OpenVINO Model Server, framework `onnx-1`.

In [ ]:
%pip install --quiet "optimum[exporters]"

In [ ]:
!optimum-cli export onnx --model {MODEL_ID} --task text-classification ./reviewiq-sentiment/1/

In [ ]:
!ls -lh ./reviewiq-sentiment/1/